ST-554 Homework 9
---
By Joshua McClure

---

Splitting the Data, Metrics, and Models
---

* Finding a data set you can fit supervised learning models with (you cannot use the datasets we’ve been using in class - there are many places with free data out there such as the UCI machine learning repository and kaggle).

* Please read your data in via a URL or include the data as a file in your submission.
https://www.kaggle.com/datasets/yasserh/titanic-dataset?select=Titanic-Dataset.csv (Link to download Dataset)

* Using a numeric or binary response, fitting three different classes of models and choosing an overall best model.

---

* Using spark MLlib, split the data into a training and test set.

* Choose and describe a metric you’ll be using to judge your models.

* You’ll be fitting three different classes of models (of your choice - they can be ones we used in class or ones we didn’t cover - see https://spark.apache.org/docs/latest/ml-classification-regression.html for a list of models they have in MLlib). Briefly describe each model (no code or anything here, just concepts and ideas about what the models are doing). These discussions should be clear to someone that knows statistics but doesn’t know the modeling type/algorithm!

---

(1) Linear Regression Model
------

It is the most simple model used for all basic mathematical and statistical pracctices. It is useful in predicting the path of the model using the best line of fit to show how certain variables affect one another. In other words showing variable realtionships and the differences between variables when compared to one another is this models strong suit.

(2) Binomial Logistic Regression Model
---

This model helps to predict the outcomes of variables with only 2 outcomes. It could be 0 and 1, True or false, In this case Survied or Died and this model is good for predicting in this case how accurately we are able to predict the Survival rate of passangers.

(3) Random Forest Model
---

It is a machine learning model that uses Bootsrapping Aggregation (Bagging) to help find the best combination of variables in a random method. Bagging makes trees where variables are assigned in randomly and tested against one another with the elimination of variables that would make the model perform worse until you get to the best model possible. This model helps to reduce overfitting (when the data training overexaggerates certain points of information to skew the data) as well as makes the model robust which makes it able to not be affected by outliers caused by misinputs or human error.

---

Model Fitting
---

Next, you should use Spark MLlib to fit your three different classes models to the training data. This
should be done using pipelines and cross validation to choose your best model for each model type. You
should compare your models using your metric chosen earlier.

* You should set up a pipeline in pyspark for each of your models

* You should do your transformations using the functions from MLlib to easily put them into the pipeline. At least one of the pipelines should use four or more transformations prior to the model fit (estimator)

  * VectorAssembler counts as a transformation

  * Doing something like a log transform counts as well

  * Adding polynomial terms or interaction terms counts

  * etc.
  
* You can use the same set of transformations for multiple models (if appropriate)

---

I decided to seperate these transformations into the three models seperately and to try to show different variables with each transformation when possible. I seperated everything into six steps for better readability.

(1) Initialize Library and Load in Dataset
---

In [12]:
# Set up Package Library
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, log as spark_log, rand, lit
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, PolynomialExpansion, SQLTransformer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator

In [13]:
# Initialize Spark Session
spark = SparkSession.builder.appName("TitanicML").getOrCreate()

# Load the dataset
df_raw = spark.read.csv("Titanic-Dataset.csv", header = True, inferSchema = True)

# Print the first few rows of the raw dataset
print("Raw Data Head:")
df_raw.show(5)

Raw Data Head:
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+---

(2) Data Preprocessing and Splitting
---

This section handles missing values, converts data types, and prepares features for model training. We'll also split the data into training and test sets.

---

In [14]:
# Drop 'Cabin' due to too many missing values
df_processed = df_raw.drop('Cabin')

# Impute missing 'Embarked' with the mode
# First, find the mode of 'Embarked'
mode_embarked = df_processed.groupBy("Embarked").count().orderBy(col("count").desc()).limit(1).collect()[0][0]
df_processed = df_processed.na.fill({'Embarked': mode_embarked})

# Impute missing 'Age' with the mean
imputer = Imputer(inputCols = ["Age"], outputCols = ["Age"], strategy = "mean")
imputer_model = imputer.fit(df_processed)
df_processed = imputer_model.transform(df_processed)

# Drop 'PassengerId', 'Name', 'Ticket' as they are not features for the model
df_processed = df_processed.drop('PassengerId', 'Name', 'Ticket')

# Cast 'Survived' to Double for MLlib models
df_processed = df_processed.withColumn("Survived", col("Survived").cast("double"))

# Print the first few rows of the processed dataset
print("Processed Data Head:")
df_processed.show(5)

# Split data into training and test sets
(trainingData, testData) = df_processed.randomSplit([0.8, 0.2], seed=42)
print(f"Training Data Count: {trainingData.count()}")
print(f"Test Data Count: {testData.count()}")

Processed Data Head:
+--------+------+------+----+-----+-----+-------+--------+
|Survived|Pclass|   Sex| Age|SibSp|Parch|   Fare|Embarked|
+--------+------+------+----+-----+-----+-------+--------+
|     0.0|     3|  male|22.0|    1|    0|   7.25|       S|
|     1.0|     1|female|38.0|    1|    0|71.2833|       C|
|     1.0|     3|female|26.0|    0|    0|  7.925|       S|
|     1.0|     1|female|35.0|    1|    0|   53.1|       S|
|     0.0|     3|  male|35.0|    0|    0|   8.05|       S|
+--------+------+------+----+-----+-----+-------+--------+
only showing top 5 rows
Training Data Count: 746
Test Data Count: 145


(3) Common Feature Transformers
---

These transformers will be used across multiple models to convert categorical features into numerical representations.

---

In [15]:
# StringIndexer for categorical features

sex_indexer = StringIndexer(inputCol = "Sex", outputCol = "Sex_indexed", handleInvalid = "keep")
embarked_indexer = StringIndexer(inputCol = "Embarked", outputCol = "Embarked_indexed", handleInvalid = "keep")

# OneHotEncoder for indexed categorical features

sex_encoder = OneHotEncoder(inputCol = "Sex_indexed", outputCol = "Sex_vec")
embarked_encoder = OneHotEncoder(inputCol = "Embarked_indexed", outputCol = "Embarked_vec")

(4) Linear Regression Model with Log Transformation
---

We will build a pipeline for Linear Regression, incorporating a log transformation on the 'Fare' feature.

---

In [16]:
# Log transformation for 'Fare'
log_fare_transformer = SQLTransformer(statement = "SELECT *, log(Fare + 0.1) AS Log_Fare FROM __THIS__")

# Feature Assembler for Linear Regression
lr_assembler = VectorAssembler(
    inputCols = ["Pclass", "Age", "SibSp", "Parch", "Log_Fare", "Sex_vec", "Embarked_vec"],
    outputCol = "features")

# Linear Regression Estimator
lr = LinearRegression(featuresCol = "features", labelCol = "Survived", solver = "normal")

# Linear Regression Pipeline
lr_pipeline = Pipeline(stages = [
    sex_indexer, embarked_indexer, sex_encoder, embarked_encoder, 
    log_fare_transformer, lr_assembler, lr
])

# Parameter Grid for Linear Regression
lr_paramGrid = ParamGridBuilder().addGrid(lr.regParam, [0.01, 0.1]).addGrid(lr.elasticNetParam, [0.0, 0.5]).build()

# CrossValidator for Linear Regression
lr_crossval = CrossValidator(
    estimator = lr_pipeline,
    estimatorParamMaps = lr_paramGrid,
    evaluator = RegressionEvaluator(labelCol = "Survived", predictionCol = "prediction", metricName = "rmse"),
    numFolds = 3, # Use 3+ folds
    seed = 42
)

(5) Binomial Regression Model with Interaction Term
---

This pipeline will use Logistic Regression (binomial) and include an interaction term between 'Age' and 'Pclass'.

---

In [17]:
# Interaction term 'Age_Pclass_Interaction'
interaction_transformer = SQLTransformer(statement = "SELECT *, Age * Pclass AS Age_Pclass_Interaction FROM __THIS__")

# Feature Assembler for Logistic Regression
bin_lr_assembler = VectorAssembler(
    inputCols = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_vec", "Embarked_vec", "Age_Pclass_Interaction"],
    outputCol = "features")

# Logistic Regression Estimator (for binomial classification)
bin_lr = LogisticRegression(featuresCol = "features", labelCol = "Survived", family = "binomial")

# Binomial Logistic Regression Pipeline
bin_lr_pipeline = Pipeline(stages = [
    sex_indexer, embarked_indexer, sex_encoder, embarked_encoder, 
    interaction_transformer, bin_lr_assembler, bin_lr
])

# Parameter Grid for Binomial Logistic Regression
bin_lr_paramGrid = ParamGridBuilder().addGrid(bin_lr.regParam, [0.01, 0.1]).addGrid(bin_lr.elasticNetParam, [0.0, 0.5]).build()

# CrossValidator for Binomial Logistic Regression
bin_lr_crossval = CrossValidator(
    estimator = bin_lr_pipeline,
    estimatorParamMaps = bin_lr_paramGrid,
    evaluator = BinaryClassificationEvaluator(labelCol = "Survived", rawPredictionCol = "rawPrediction", metricName = "areaUnderROC"),
    numFolds = 3, # Use 3+ folds in practice
    seed = 42
)

(6) Random Forest Model with Polynomial Term
---

This pipeline will implement a Random Forest Classifier with polynomial features generated from 'Age'.

---

In [18]:
# Feature Assembler for Random Forest
# Define the final assembler to include polynomial features
rf_assembler = VectorAssembler(
    inputCols = ["Pclass", "SibSp", "Parch", "Fare", "Sex_vec", "Embarked_vec", "Age_poly"],
    outputCol = "features")

# Random Forest Classifier Estimator
rf = RandomForestClassifier(featuresCol = "features", labelCol = "Survived", seed = 42)

# Random Forest Pipeline
rf_pipeline = Pipeline(stages = [
    sex_indexer, embarked_indexer, sex_encoder, embarked_encoder,
    VectorAssembler(inputCols = ["Age"], outputCol = "Age_vector_for_poly"), # Vectorize Age first
    PolynomialExpansion(inputCol = "Age_vector_for_poly", outputCol = "Age_poly", degree = 2),
    rf_assembler, rf 
])

# Parameter Grid for Random Forest
rf_paramGrid = ParamGridBuilder().addGrid(rf.numTrees, [10, 20]).addGrid(rf.maxDepth, [5, 10]).build()

# CrossValidator for Random Forest
rf_crossval = CrossValidator(
    estimator = rf_pipeline,
    estimatorParamMaps = rf_paramGrid,
    evaluator = BinaryClassificationEvaluator(labelCol = "Survived", rawPredictionCol = "rawPrediction", metricName = "areaUnderROC"),
    numFolds = 3, # Use 3+ folds
    seed = 42
)

Model Testing
---

Lastly, you should evaluate the best models from each class on the test set and state which overall model
was deemed the best.

---

In [19]:
# 1. Fit models using CrossValidator on training data

print("Fitting Linear Regression Model...")
lr_model = lr_crossval.fit(trainingData)
print("Fitting Binomial Logistic Regression Model...")
bin_lr_model = bin_lr_crossval.fit(trainingData)
print("Fitting Random Forest Model...")
rf_model = rf_crossval.fit(trainingData)

# 2. Make predictions on the test set

lr_predictions = lr_model.transform(testData)
bin_lr_predictions = bin_lr_model.transform(testData)
rf_predictions = rf_model.transform(testData)

# 3. Evaluate predictions

# Linear Regression Evaluation (RMSE)
lr_evaluator = RegressionEvaluator(labelCol="Survived", predictionCol="prediction", metricName="rmse")
lr_rmse = lr_evaluator.evaluate(lr_predictions)
print(f"Linear Regression RMSE on Test Data: {lr_rmse}")

# Binomial Logistic Regression Evaluation (Area Under ROC)
bin_lr_evaluator = BinaryClassificationEvaluator(labelCol="Survived", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
bin_lr_auc = bin_lr_evaluator.evaluate(bin_lr_predictions)
print(f"Binomial Logistic Regression Area Under ROC on Test Data: {bin_lr_auc}")

# Random Forest Evaluation (Area Under ROC)
rf_evaluator = BinaryClassificationEvaluator(labelCol="Survived", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
rf_auc = rf_evaluator.evaluate(rf_predictions)
print(f"Random Forest Area Under ROC on Test Data: {rf_auc}")

# 4. Compare results and determine the best model
print("\n--- Model Comparison ---")
print(f"Linear Regression (RMSE): {lr_rmse}")
print(f"Binomial Logistic Regression (AUC): {bin_lr_auc}")
print(f"Random Forest (AUC): {rf_auc}")

Fitting Linear Regression Model...


26/04/14 18:43:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Fitting Binomial Logistic Regression Model...
Fitting Random Forest Model...
Linear Regression RMSE on Test Data: 0.3706733071864193
Binomial Logistic Regression Area Under ROC on Test Data: 0.855679156908665
Random Forest Area Under ROC on Test Data: 0.8882708821233412

--- Model Comparison ---
Linear Regression (RMSE): 0.3706733071864193
Binomial Logistic Regression (AUC): 0.855679156908665
Random Forest (AUC): 0.8882708821233412

Among the classification models, Random Forest performs best with an AUC of 0.8883.
The Linear Regression model provided an RMSE of 0.3707, which is a different metric and not directly comparable for 'best' classification performance.


Interpretation
---

Out of the three models presented the Random Forest Model has an Area Under the Curve (AUC) of .889. Measuring the AUC shows that the model will have a better ability to distinguah between positive an negative cases. These Positove and Negative cases when paired with ROC curve also distiguishes between Positives and False Positives. So the model has around a 90% chance in correctly measuring the individuals who survived the Titanic.